# Eulerian Splatting: train → render → evaluate (with live progress)

Select the **`eulerian-splatting (B200, cu128)`** kernel. It already sets up CUDA 12.8 and `ninja`, so no `module load` is needed.
If the kernel is missing, register it once from a terminal:
```bash
/blue/egn5216/Photo_Gauss_Test/envs/eulerian/bin/python -m ipykernel install --user --name eulerian-splatting \
  --display-name "eulerian-splatting (B200, cu128)" \
  --env PATH "/blue/egn5216/Photo_Gauss_Test/envs/eulerian/bin:/apps/compilers/cuda/12.8.1/bin:/usr/local/bin:/usr/bin:/bin" \
  --env CUDA_HOME /apps/compilers/cuda/12.8.1 --env LD_LIBRARY_PATH "/apps/compilers/cuda/12.8.1/lib64:/opt/slurm/lib64"
```

During training, the dashboard shows:
- **Header:** phase (probabilistic / refinement), iteration, it/s, ETA for training alone and with evals and saves, estimated finish time, a per-stage GPU time breakdown for the last iteration (sample / render / loss / backward / step), loss, the number of Gaussians, GPU memory, and **host RAM against the SLURM job limit** (it turns red near the limit, before an OOM kill).
- **Plots:** the current training render, its GT, and an error map; the loss curve; PSNR at each eval; stacked per-stage time and the Gaussian count over training.
- **Test grid** from the latest eval, plus an **event log** (setup stages, evals, saves, the switch to refinement).

## 0. Environment check

In [ ]:
import os, sys, time, json, glob
REPO = "/blue/egn5216/Photo_Gauss_Test/eulerian-splatting"
os.chdir(REPO); sys.path.insert(0, REPO)

import torch
from utils.notebook_progress import _cgroup_memory, _fmt_bytes
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA (torch)", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU visible: start this Jupyter session on a GPU node (hpg-b200)."
name = torch.cuda.get_device_name(0); cc = torch.cuda.get_device_capability(0)
print(f"GPU: {name}  (sm_{cc[0]}{cc[1]}), {_fmt_bytes(torch.cuda.mem_get_info()[1])}")
print("CUDA_HOME:", os.environ.get("CUDA_HOME"), "| ninja on PATH:", any(os.path.exists(os.path.join(p, "ninja")) for p in os.environ["PATH"].split(":")))
cg = _cgroup_memory()
if cg:
    print(f"Host RAM (SLURM job): {_fmt_bytes(cg[0])} used / {_fmt_bytes(cg[1])} limit")
    if cg[1] < 16 * 2**30:
        print("  ⚠ Under 16 GB. Full-res image loading and PLY saving can get OOM-killed; prefer images_4 + resolution=1 below.")
ARCH = f"{cc[0]}.{cc[1]}"   # compile the sampling kernel for exactly this GPU (10.0 on B200)

## 1. Run settings
`QUICK_TEST = True` runs a short schedule (a few minutes) to check that everything works end to end. Set it to `False` for the paper schedule (30K + 5K iterations for bicycle).

In [ ]:
CONFIG      = "bicycle"                 # resolves to configs/scenes/<CONFIG>.yaml
SOURCE_PATH = "data/bicycle"
QUICK_TEST  = True

common = dict(
    config=CONFIG,
    source_path=SOURCE_PATH,
    torch_cuda_arch_list=ARCH,
    dry_run=True,                      # False -> log to wandb
    disable_visualize=True,            # the dashboard shows the test grid instead of plt.show()
    # Read the pre-downsampled images instead of decoding full-res and resizing (same result, much less host RAM).
    # Equivalent to the YAML's resolution: 4.
    images="images_4", resolution=1,
)
if QUICK_TEST:
    common.update(iterations=1000, refinement_iterations=300, save_iterations=[1000, 1300], test_interval=500,
                  num_samples=5_000_000, num_samples_min=2_000_000)   # fewer Gaussians -> smaller PLY saves (host RAM)

os.makedirs(os.path.join(SOURCE_PATH, "point_cloud"), exist_ok=True)   # required by random init
common

## 2. Import (the first run JIT-compiles the Slang rasterizer kernels)
On a fresh machine this takes a few minutes. Later runs hit the cache in `~/.cache/torch_extensions`.

In [ ]:
t = time.perf_counter()
from run import train_model, render_model, evaluate_model
from utils.notebook_progress import TrainingDashboard
print(f"Imports + kernel compile/load: {time.perf_counter() - t:.1f}s")

## 3. Training (live dashboard)
`refresh_s` sets how often the dashboard redraws. It reads GPU timings and copies the preview images only when it redraws, so the overhead is negligible.
Expect the first ~300 iterations to read slow: they include the one-time compile of the CUDA sampling kernel.

In [ ]:
dash = TrainingDashboard(refresh_s=5)
t = time.perf_counter()
model_path = train_model(progress_callback=dash, **common)
print(f"Training wall time: {time.perf_counter() - t:.0f}s  ->  {model_path}")

## 4. Rendering
Renders the frozen final sample (post-refinement) for the train and test views. Pass `eval_iteration=<iterations>` to get the pre-refinement numbers instead (paper Table 1). The progress bar shows the ETA.

In [ ]:
# model_path = "output/bicycle/<run>"   # uncomment to render an earlier run
t = time.perf_counter()
render_model(config=CONFIG, model_path=model_path, source_path=SOURCE_PATH, images=common["images"], resolution=common["resolution"],
             eval_iteration=-1, skip_train=True)
print(f"Rendering: {time.perf_counter() - t:.0f}s")

In [ ]:
# Side-by-side preview of a few test renders
import matplotlib.pyplot as plt
from PIL import Image as PILImage
run_dir = sorted(glob.glob(os.path.join(model_path, "renders", "test", "ours_*")), key=lambda p: int(p.split("_")[-1]))[-1]
files = sorted(glob.glob(os.path.join(run_dir, "renders", "*.png")))[:4]
fig, axs = plt.subplots(len(files), 2, figsize=(12, 3.6 * len(files)), squeeze=False)
for row, f in zip(axs, files):
    row[0].imshow(PILImage.open(f)); row[0].set_title("render " + os.path.basename(f))
    row[1].imshow(PILImage.open(os.path.join(run_dir, "gt", os.path.basename(f)))); row[1].set_title("GT")
    for a in row: a.axis("off")
plt.suptitle(os.path.basename(run_dir)); plt.tight_layout(); plt.show()

## 5. Evaluation (PSNR / SSIM / LPIPS on the rendered test views)

In [ ]:
t = time.perf_counter()
evaluate_model([model_path])
print(f"Metrics: {time.perf_counter() - t:.0f}s")
from IPython.display import HTML, display
res = json.load(open(os.path.join(model_path, "results.json")))
rows = "".join(f"<tr><td>{m}</td>" + "".join(f"<td>{v:.4f}</td>" for v in d.values()) + "</tr>" for m, d in res.items())
hdr = "".join(f"<th>{k}</th>" for k in next(iter(res.values())).keys())
display(HTML(f"<table><tr><th>method</th>{hdr}</tr>{rows}</table>"))